In [1]:
# # remember to resteart kernell
# ! pip install -e ../../wazeasy
# ! pip install pandas "dask[complete]" pyarrow plotly
# ! pip install --upgrade s3fs
# ! pip install geopandas
# ! pip install seaborn
# ! pip install folium matplotlib mapclassifys
# ! pip install h3
# ! pip install dask-geopandas
# ! pip install folium matplotlib mapclassify

# # !rm -rf /tmp/*

In [2]:
import os
os.environ['AWS_CA_BUNDLE'] = '/Users/mtadeo/Documents/gitrepo/cacert_new.pem'
os.environ['REQUESTS_CA_BUNDLE'] = '/Users/mtadeo/Documents/gitrepo/cacert_new.pem'

In [3]:
import pandas as pd
import dask.dataframe as dd
import dask_geopandas
from dask import delayed, compute
from wazeasy import utils, plots, reports
import geopandas as gpd
from shapely import Polygon
from h3 import LatLngPoly
import yaml
import altair as alt
alt.renderers.enable('mimetype')
import re

In [4]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [5]:
storage_options = {'profile': 'ddp_dec',
    # 'client_kwargs': {
    #     'verify': '/Users/mtadeo/Documents/gitrepo/cacert_new.pem'
    #     }
                  }

file_list = [
    's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000000.parquet',
    's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000001.parquet',
    's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000002.parquet',
    's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000003.parquet',
    's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000004.parquet',
    's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000005.parquet',
    's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000006.parquet',
    's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000007.parquet',
    's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000008.parquet',
    's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000009.parquet'
]
# ddf = dd.read_parquet(file_list, storage_options = storage_options, engine = 'pyarrow')
# ddf = dd.read_parquet('s3://wbg-waze/bq/IQ/682baghdad/jams/*.parquet', 
#                       storage_options = storage_options, engine = 'pyarrow')

# path = 's3://wbg-waze/bq/IQ/682baghdad/jams/'
# df = pd.read_parquet(path, storage_options=storage_options)

In [6]:
%%time
ddf = utils.load_data(file_list, 
                      storage_options = storage_options, file_type = 'parquet', 
                      filter_level_5 = True, 
                      usecols = ['ts', 'geoWKT', 'uuid', 'level','length'])
ddf = ddf.repartition(npartitions=80)
# ddf = ddf.repartition(npartitions=400)
# ddf = ddf.persist()
# ddf = ddf.compute()


CPU times: user 555 ms, sys: 139 ms, total: 694 ms
Wall time: 8.28 s


In [7]:
timezone = config['Baghdad']['timezone']['timezone_name']
geographies = config['Baghdad']['geographies']
projected_crs = config['Baghdad']['crs']['projected_crs']

In [8]:
dict_geogs = {geog: gpd.read_file(geog_data['path']) for geog, geog_data in geographies.items()}

In [9]:
ddf = utils.assign_geography_to_jams(ddf, projected_crs)

In [10]:
utils.handle_time(ddf, timezone)

In [11]:
# reports.run_basic_report(ddf, '2023-01-01', '2024-12-31')

In [12]:
%%time
ddf = utils.assign_geography_to_jams(ddf, projected_crs, dict_geogs)

CPU times: user 29.9 s, sys: 6.25 s, total: 36.2 s
Wall time: 1min 4s


In [14]:
%%time
ddf = ddf.persist()

CPU times: user 8.49 s, sys: 2.86 s, total: 11.3 s
Wall time: 21.7 s


In [14]:
reports.run_geog_report(ddf, geographies, '2023-01-01', '2024-12-31')

Running report for Administrative Level 1


<VegaLite 6 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


<VegaLite 6 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


<VegaLite 6 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


Running report for Administrative Level 2


<VegaLite 6 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


<VegaLite 6 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


<VegaLite 6 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


Running report for Hexagons Level 7


# Accidents